In [1]:
import os
import gc
import random
import math
import json
import shutil
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import time
from pathlib import Path
from collections import defaultdict
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    Activation,
    MaxPooling2D,
    UpSampling2D,
    Concatenate,
    Add
)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
PREPARED = Path("/kaggle/input/datasets/swaratakale/dip-project-prepared-dataset")
DATASET = PREPARED / "dataset"

PROJECT = Path("/kaggle/working/DIP_Project")
CHECKPOINTS = PROJECT / "checkpoints"
RESULTS = PROJECT / "results"
PREDICTIONS = PROJECT / "predictions"

for folder in [CHECKPOINTS, RESULTS, PREDICTIONS]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATASET / "train" / "Outputs"
VAL_INPUT_DIR = DATASET / "validation" / "Inputs"
VAL_OUTPUT_DIR = DATASET / "validation" / "Outputs"
TEST_INPUT_DIR = DATASET / "test" / "Inputs"
TEST_OUTPUT_DIR = DATASET / "test" / "Outputs"

train_files = sorted(TRAIN_DIR.glob("*"))
val_inputs = sorted(VAL_INPUT_DIR.glob("*"))
val_outputs = sorted(VAL_OUTPUT_DIR.glob("*"))

print("Training images:", len(train_files))
print("Validation inputs:", len(val_inputs))
print("Validation targets:", len(val_outputs))

assert len(train_files) > 0
assert len(val_inputs) == len(val_outputs) > 0

Training images: 200
Validation inputs: 25
Validation targets: 25


In [3]:
PATCH_SIZE = 352

def make_training_pair(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        raise ValueError(f"Could not read {image_path}")

    h, w = image.shape
    scale = max(
        PATCH_SIZE / min(h, w),
        random.uniform(0.75, 1.25)
    )

    new_w = max(PATCH_SIZE, int(w * scale))
    new_h = max(PATCH_SIZE, int(h * scale))

    image = cv2.resize(image, (new_w, new_h))

    angle = random.uniform(0, 360)
    matrix = cv2.getRotationMatrix2D(
        (new_w / 2, new_h / 2), angle, 1
    )
    image = cv2.warpAffine(
        image, matrix, (new_w, new_h), borderValue=255
    )

    if random.random() < 0.5:
        image = cv2.flip(image, 1)

    if random.random() < 0.5:
        image = cv2.flip(image, 0)

    h, w = image.shape
    x = random.randint(0, w - PATCH_SIZE)
    y = random.randint(0, h - PATCH_SIZE)

    clean = image[y:y+PATCH_SIZE, x:x+PATCH_SIZE].copy()
    damaged = clean.copy()

    ys, xs = np.where(clean < 128)
    num_squares = random.randint(10, 20)

    if len(xs) > 0:
        for _ in range(num_squares):
            side = random.randint(10, 50)
            idx = random.randrange(len(xs))

            cx, cy = int(xs[idx]), int(ys[idx])

            x1 = max(0, min(cx - side // 2, PATCH_SIZE - side))
            y1 = max(0, min(cy - side // 2, PATCH_SIZE - side))

            damaged[y1:y1+side, x1:x1+side] = 255

    foreground = clean < 128
    erased = damaged > clean

    weights = np.ones(clean.shape, dtype=np.float32)
    weights[foreground & erased] = 5.0
    weights[foreground & ~erased] = 2.0

    clean = clean.astype(np.float32) / 255.0
    damaged = damaged.astype(np.float32) / 255.0

    return damaged[..., None], clean[..., None], weights


def training_generator(batch_size):
    while True:
        batch_x, batch_y, batch_w = [], [], []

        for _ in range(batch_size):
            image_path = random.choice(train_files)
            damaged, clean, weights = make_training_pair(image_path)

            batch_x.append(damaged)
            batch_y.append(clean)
            batch_w.append(weights)

        yield (
            np.stack(batch_x),
            np.stack(batch_y),
            np.stack(batch_w)
        )

In [4]:
x, y, w = next(training_generator(2))

print("Input:", x.shape)
print("Target:", y.shape)
print("Weights:", w.shape)
print("Unique weights:", np.unique(w))

assert x.shape == (2, 352, 352, 1)
assert y.shape == (2, 352, 352, 1)
assert w.shape == (2, 352, 352)

Input: (2, 352, 352, 1)
Target: (2, 352, 352, 1)
Weights: (2, 352, 352)
Unique weights: [1. 2. 5.]


In [5]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(
        filters, kernel_size,
        strides=strides,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


def build_model1():
    inputs = Input(shape=(None, None, 1))

    x = conv_bn_relu(inputs, 24, kernel_size=5, strides=2)
    x = conv_bn_relu(x, 64, strides=2)
    x = conv_bn_relu(x, 128)
    x = conv_bn_relu(x, 256, strides=2)
    x = conv_bn_relu(x, 512)
    x = conv_bn_relu(x, 512, strides=2)
    x = conv_bn_relu(x, 256)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 128)
    x = conv_bn_relu(x, 64)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 32)
    x = conv_bn_relu(x, 16)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 8)
    x = conv_bn_relu(x, 4)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 2)
    x = conv_bn_relu(x, 1)

    outputs = layers.Conv2D(
        1, 3, padding="same", activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Paper_Encoder_Decoder")


def conv_bn_relu_dilated(x, filters, kernel_size=3, dilation_rate=1):
    x = layers.Conv2D(
        filters, kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


def build_model2():
    inputs = Input(shape=(None, None, 1))

    x = conv_bn_relu_dilated(inputs, 64, kernel_size=11)
    x = conv_bn_relu_dilated(x, 128)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=1)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=2)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=4)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=8)
    x = conv_bn_relu_dilated(x, 128)
    x = conv_bn_relu_dilated(x, 64)

    outputs = layers.Conv2D(
        1, 3, padding="same", activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Dilated_CNN")


model1 = build_model1()
model2 = build_model2()

print("Model 1 parameters:", model1.count_params())
print("Model 2 parameters:", model2.count_params())

Model 1 parameters: 5502904
Model 2 parameters: 2520705


In [6]:
CHECKPOINT_INTERVAL = 5000

def find_latest_checkpoint(model_name):
    files = list(CHECKPOINTS.glob(f"{model_name}_iter_*.keras"))

    if not files:
        return None, 0

    def get_iteration(path):
        match = re.search(r"_iter_(\d+)\.keras$", path.name)
        return int(match.group(1)) if match else 0

    latest = max(files, key=get_iteration)
    return latest, get_iteration(latest)


class CheckpointCallback(tf.keras.callbacks.Callback):
    def __init__(self, model_name, start_iteration, interval=5000):
        super().__init__()
        self.model_name = model_name
        self.start_iteration = start_iteration
        self.interval = interval

    def on_train_batch_end(self, batch, logs=None):
        iteration = self.start_iteration + batch + 1

        if iteration % self.interval == 0:
            path = CHECKPOINTS / (
                f"{self.model_name}_iter_{iteration:08d}.keras"
            )
            self.model.save(path)
            print(f"\nSaved checkpoint: {path.name}")


def train_experiment(
    model_name,
    model_builder,
    batch_size,
    iterations=5000
):
    latest, start_iteration = find_latest_checkpoint(model_name)

    if latest:
        print("Resuming from:", latest.name)
        model = tf.keras.models.load_model(latest)
    else:
        print("Starting new model:", model_name)
        model = model_builder()
        model.compile(
            optimizer=tf.keras.optimizers.Adadelta(),
            loss="mse"
        )

    print("Starting iteration:", start_iteration + 1)
    print("Iterations to run:", iterations)

    generator = training_generator(batch_size)
    callback = CheckpointCallback(
        model_name,
        start_iteration,
        CHECKPOINT_INTERVAL
    )

    start_time = time.time()

    history = model.fit(
        generator,
        steps_per_epoch=iterations,
        epochs=1,
        callbacks=[callback],
        verbose=1
    )

    total_iterations = start_iteration + iterations

    final_path = CHECKPOINTS / (
        f"{model_name}_iter_{total_iterations:08d}.keras"
    )

    if not final_path.exists():
        model.save(final_path)
        print("Saved final checkpoint:", final_path.name)

    print("\nTraining completed!")
    print("Total iterations:", total_iterations)
    print("Final weighted loss:", history.history["loss"][-1])
    print("Time (minutes):", round((time.time() - start_time) / 60, 2))

    return model, total_iterations

In [7]:
tf.keras.backend.clear_session()
gc.collect()

model1_weighted, model1_iterations = train_experiment(
    model_name="Model1_weighted",
    model_builder=build_model1,
    batch_size=4,
    iterations=50000
)

Starting new model: Model1_weighted


Starting iteration: 1
Iterations to run: 50000


E0000 00:00:1794780464.940355      73 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


    1/50000 ━━━━━━━━━━━━━━━━━━━━ 432:44:11 31s/step - loss: 0.2752

    2/50000 ━━━━━━━━━━━━━━━━━━━━ 1:21:04 97ms/step - loss: 0.2798 

    3/50000 ━━━━━━━━━━━━━━━━━━━━ 3:24:48 246ms/step - loss: 0.2813

... (49,995 outputs left out)

49999/50000 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.2440


Saved checkpoint: Model1_weighted_iter_00050000.keras
50000/50000 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.2440

50000/50000 ━━━━━━━━━━━━━━━━━━━━ 3425s 68ms/step - loss: 0.2269



Training completed!
Total iterations: 50000
Final weighted loss: 0.2269212305545807
Time (minutes): 57.09


In [8]:
del model1_weighted

tf.keras.backend.clear_session()
gc.collect()

0

In [9]:
model2_weighted, model2_iterations = train_experiment(
    model_name="Model2_weighted",
    model_builder=build_model2,
    batch_size=2,
    iterations=50000
)

Starting new model: Model2_weighted
Starting iteration: 1
Iterations to run: 50000


E0000 00:00:1794783909.942935      73 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


    1/50000 ━━━━━━━━━━━━━━━━━━━━ 1365:22:13 98s/step - loss: 0.3343

    2/50000 ━━━━━━━━━━━━━━━━━━━━ 11:28:00 826ms/step - loss: 0.3245

    3/50000 ━━━━━━━━━━━━━━━━━━━━ 11:21:15 818ms/step - loss: 0.3204

    4/50000 ━━━━━━━━━━━━━━━━━━━━ 11:25:00 822ms/step - loss: 0.3181

... (48,756 outputs left out)

48761/50000 ━━━━━━━━━━━━━━━━━━━━ 16:46 813ms/step - loss: 0.0705

48762/50000 ━━━━━━━━━━━━━━━━━━━━ 16:45 813ms/step - loss: 0.0705

48763/50000 ━━━━━━━━━━━━━━━━━━━━ 16:45 813ms/step - loss: 0.0705

48764/50000 ━━━━━━━━━━━━━━━━━━━━ 16:44 813ms/step - loss: 0.0705

In [ ]:

# import zipfile
# import gc

# # Location of the attached checkpoint dataset
# CHECKPOINT_DATASET = Path(
#     "/kaggle/input/datasets/swaratakale/trained-checkpoints"
# )

# # Start with a clean output checkpoint directory
# for f in CHECKPOINTS.glob("Model*_iter_*.keras"):
#     f.unlink()

# # Restore the 10,000-iteration checkpoints
# for name in ["Model1", "Model2"]:
#     source = CHECKPOINT_DATASET / f"{name}_iter_00010000"
#     destination = CHECKPOINTS / f"{name}_iter_00010000.keras"

#     assert source.exists(), f"Missing checkpoint folder: {source}"

#     with zipfile.ZipFile(
#         destination, "w", zipfile.ZIP_DEFLATED
#     ) as z:
#         for filename in [
#             "config.json",
#             "metadata.json",
#             "model.weights.h5"
#         ]:
#             file_path = source / filename
#             assert file_path.exists(), f"Missing: {file_path}"
#             z.write(file_path, arcname=filename)

#     print("Restored:", destination.name)

# # Release unused model objects
# del model1, model2
# tf.keras.backend.clear_session()
# gc.collect()

# # MODEL 1: 10,000 -> 50,000
# model1, model1_iterations = train_experiment(
#     model_name="Model1",
#     model_builder=build_model1,
#     batch_size=4,
#     iterations=40000,
#     checkpoint_interval=10000
# )

# del model1
# tf.keras.backend.clear_session()
# gc.collect()

# # MODEL 2: 10,000 -> 50,000
# model2, model2_iterations = train_experiment(
#     model_name="Model2",
#     model_builder=build_model2,
#     batch_size=2,
#     iterations=40000,
#     checkpoint_interval=10000
# )

# print("\nBOTH EXPERIMENTS COMPLETED!")
# print("Model 1:", model1_iterations)
# print("Model 2:", model2_iterations)


In [ ]:
# import zipfile
# import gc
# import tensorflow as tf
# from pathlib import Path

# CHECKPOINT_DATASET_50K = Path(
#     "/kaggle/input/datasets/swaratakale/trained-checkpoints-50000"
# )

# CHECKPOINTS = Path("/kaggle/working/DIP_Project/checkpoints")
# CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# for name in ["Model1", "Model2"]:

#     source = CHECKPOINT_DATASET_50K / f"{name}_iter_00050000"
#     destination = CHECKPOINTS / f"{name}_iter_00050000.keras"

#     assert source.exists(), f"Missing folder: {source}"

#     with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as z:
#         for filename in [
#             "config.json",
#             "metadata.json",
#             "model.weights.h5"
#         ]:
#             file_path = source / filename
#             assert file_path.exists(), f"Missing file: {file_path}"
#             z.write(file_path, arcname=filename)

#     print(
#         f"{name}: restored successfully, "
#         f"{destination.stat().st_size / (1024**2):.2f} MB"
#     )

In [ ]:
# model1 = tf.keras.models.load_model(
#     CHECKPOINTS / "Model1_iter_00050000.keras"
# )

# print("Model 1 loaded!")
# print("Parameters:", model1.count_params())

In [ ]:
# import numpy as np
# import cv2

# PATCH_SIZE = 352
# OVERLAP = 64
# STRIDE = PATCH_SIZE - OVERLAP

# def predict_tiled(model, image):
#     """
#     Predict a full image using overlapping patches
#     and weighted blending.
#     """
#     h, w = image.shape

#     image = image.astype(np.float32) / 255.0

#     pad_h = max(0, PATCH_SIZE - h)
#     pad_w = max(0, PATCH_SIZE - w)

#     padded = np.pad(
#         image,
#         ((0, pad_h), (0, pad_w)),
#         mode="reflect"
#     )

#     H, W = padded.shape

#     ys = list(range(0, max(1, H - PATCH_SIZE + 1), STRIDE))
#     xs = list(range(0, max(1, W - PATCH_SIZE + 1), STRIDE))

#     if ys[-1] != H - PATCH_SIZE:
#         ys.append(H - PATCH_SIZE)

#     if xs[-1] != W - PATCH_SIZE:
#         xs.append(W - PATCH_SIZE)

#     # Smooth blending weights
#     blend = np.outer(
#         np.hanning(PATCH_SIZE),
#         np.hanning(PATCH_SIZE)
#     ).astype(np.float32)

#     blend = np.maximum(blend, 1e-3)

#     prediction_sum = np.zeros((H, W), dtype=np.float32)
#     weight_sum = np.zeros((H, W), dtype=np.float32)

#     for y in ys:
#         for x in xs:
#             patch = padded[
#                 y:y+PATCH_SIZE,
#                 x:x+PATCH_SIZE
#             ]

#             batch = patch[None, ..., None]

#             pred = model.predict(batch, verbose=0)[0, ..., 0]

#             prediction_sum[
#                 y:y+PATCH_SIZE,
#                 x:x+PATCH_SIZE
#             ] += pred * blend

#             weight_sum[
#                 y:y+PATCH_SIZE,
#                 x:x+PATCH_SIZE
#             ] += blend

#     prediction = prediction_sum / np.maximum(weight_sum, 1e-8)

#     return prediction[:h, :w]

In [ ]:
# from pathlib import Path
# import cv2

# DATASET = Path("/kaggle/input/datasets/swaratakale/dip-project-prepared-dataset/dataset")

# val_inputs = sorted((DATASET / "validation" / "Inputs").glob("*"))
# val_outputs = sorted((DATASET / "validation" / "Outputs").glob("*"))

# print("Validation inputs:", len(val_inputs))
# print("Validation outputs:", len(val_outputs))
# print("First image:", val_inputs[0].name)

In [ ]:
# val_input = cv2.imread(
#     str(val_inputs[1]),
#     cv2.IMREAD_GRAYSCALE
# )

# prediction1 = predict_tiled(model1, val_input)

# print("Prediction shape:", prediction1.shape)
# print("Minimum:", prediction1.min())
# print("Maximum:", prediction1.max())

In [ ]:
# del model1
# tf.keras.backend.clear_session()
# gc.collect()

# model2 = tf.keras.models.load_model(
#     CHECKPOINTS / "Model2_iter_00050000.keras"
# )

# print("Model 2 loaded!")
# print("Parameters:", model2.count_params())

In [ ]:
# prediction2 = predict_tiled(model2, val_input)

# print("Prediction shape:", prediction2.shape)
# print("Minimum:", prediction2.min())
# print("Maximum:", prediction2.max())

In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 4, figsize=(20, 6))

# axes[0].imshow(val_input, cmap="gray", vmin=0, vmax=255)
# axes[0].set_title("Damaged Input")

# axes[1].imshow(prediction1, cmap="gray", vmin=0, vmax=1)
# axes[1].set_title("Model 1: Encoder-Decoder")

# axes[2].imshow(prediction2, cmap="gray", vmin=0, vmax=1)
# axes[2].set_title("Model 2: Dilated CNN")

# ground_truth = cv2.imread(
#     str(val_outputs[1]),
#     cv2.IMREAD_GRAYSCALE
# )

# axes[3].imshow(ground_truth, cmap="gray", vmin=0, vmax=255)
# axes[3].set_title("Ground Truth")

# for ax in axes:
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
# import numpy as np
# import cv2
# import random

# # One clean training image
# train_clean = cv2.imread(
#     str(train_files[0]),
#     cv2.IMREAD_GRAYSCALE
# )

# # Generate one training pair using the ACTUAL training function
# train_damaged, train_target, train_weights = make_training_pair(train_files[0])

# train_damage_ratio = np.mean(
#     train_damaged[..., 0] > train_target[..., 0]
# )

# # One validation pair
# val_clean = cv2.imread(
#     str(val_outputs[0]),
#     cv2.IMREAD_GRAYSCALE
# )

# val_damaged = cv2.imread(
#     str(val_inputs[0]),
#     cv2.IMREAD_GRAYSCALE
# )

# val_damage_ratio = np.mean(
#     val_damaged > val_clean
# )

# print("Training patch damage ratio:", round(train_damage_ratio * 100, 3), "%")
# print("Validation image damage ratio:", round(val_damage_ratio * 100, 3), "%")

# print("\nTraining patch shape:", train_target.shape)
# print("Validation image shape:", val_clean.shape)

# print("Training weights shape:", train_weights.shape)
# print("Unique weights:", np.unique(train_weights, return_counts=True))

In [ ]:
# import cv2
# import numpy as np

# def pixel_statistics(image_files, count=50):
#     black_ratios = []
#     white_ratios = []

#     for path in image_files[:count]:
#         image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

#         black = np.mean(image < 128)
#         white = np.mean(image >= 128)

#         black_ratios.append(black)
#         white_ratios.append(white)

#     print("Average black pixels:", round(np.mean(black_ratios) * 100, 2), "%")
#     print("Average white pixels:", round(np.mean(white_ratios) * 100, 2), "%")

# print("TRAINING DATA")
# pixel_statistics(train_files)

# print("\nVALIDATION TARGETS")
# pixel_statistics(val_outputs)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# ratios = []

# for path in train_files:
#     image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

#     # Estimate foreground proportion
#     ratio = np.mean(image < 128)
#     ratios.append(ratio * 100)

# ratios = np.array(ratios)

# print("Minimum foreground:", round(ratios.min(), 2), "%")
# print("Median foreground:", round(np.median(ratios), 2), "%")
# print("Maximum foreground:", round(ratios.max(), 2), "%")

# plt.figure(figsize=(9, 4))
# plt.hist(ratios, bins=20)
# plt.xlabel("Black pixel percentage")
# plt.ylabel("Number of images")
# plt.title("Foreground distribution in training images")
# plt.show()

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# patch_ratios = []

# for _ in range(500):
#     image_path = random.choice(train_files)

#     damaged, clean = make_training_pair(image_path)

#     # Black pixels in the clean target patch
#     ratio = np.mean(clean[..., 0] < (128 / 255))

#     patch_ratios.append(ratio * 100)

# patch_ratios = np.array(patch_ratios)

# print("Sampled patches:", len(patch_ratios))
# print("Minimum foreground:", round(patch_ratios.min(), 2), "%")
# print("Median foreground:", round(np.median(patch_ratios), 2), "%")
# print("Mean foreground:", round(patch_ratios.mean(), 2), "%")
# print("Maximum foreground:", round(patch_ratios.max(), 2), "%")

# plt.figure(figsize=(9, 4))
# plt.hist(patch_ratios, bins=25)
# plt.xlabel("Black pixel percentage per patch")
# plt.ylabel("Number of patches")
# plt.title("Foreground distribution in actual training patches")
# plt.show()

In [ ]:
# erased_line_ratios = []
# erased_area_ratios = []

# for _ in range(500):
#     image_path = random.choice(train_files)

#     damaged, clean = make_training_pair(image_path)

#     # Pixels that were originally black and subsequently erased
#     erased_lines = (
#         (clean[..., 0] < 128 / 255) &
#         (damaged[..., 0] >= 0.999)
#     )

#     erased_line_ratios.append(
#         np.mean(erased_lines) * 100
#     )

#     erased_area_ratios.append(
#         np.mean(damaged[..., 0] > clean[..., 0]) * 100
#     )

# print(
#     "Average erased foreground pixels:",
#     round(np.mean(erased_line_ratios), 3), "%"
# )

# print(
#     "Average erased area:",
#     round(np.mean(erased_area_ratios), 3), "%"
# )

# print(
#     "Patches with no erased foreground:",
#     sum(x == 0 for x in erased_line_ratios),
#     "/ 500"
# )

In [ ]:
# erased_fractions = []

# for _ in range(500):
#     image_path = random.choice(train_files)
#     damaged, clean = make_training_pair(image_path)

#     original_lines = clean[..., 0] < (128 / 255)
#     erased_lines = original_lines & (damaged[..., 0] >= 0.999)

#     total_lines = np.sum(original_lines)
#     erased = np.sum(erased_lines)

#     if total_lines > 0:
#         erased_fractions.append(100 * erased / total_lines)

# print("Average fraction of original line pixels erased:",
#       round(np.mean(erased_fractions), 2), "%")

# print("Median fraction erased:",
#       round(np.median(erased_fractions), 2), "%")

In [ ]:
# val_erased_fractions = []
# val_damaged_areas = []

# for input_path, output_path in zip(val_inputs, val_outputs):
#     damaged = cv2.imread(str(input_path), cv2.IMREAD_GRAYSCALE)
#     clean = cv2.imread(str(output_path), cv2.IMREAD_GRAYSCALE)

#     original_lines = clean < 128
#     erased_lines = original_lines & (damaged >= 250)

#     total_lines = np.sum(original_lines)
#     erased = np.sum(erased_lines)

#     if total_lines > 0:
#         val_erased_fractions.append(100 * erased / total_lines)

#     val_damaged_areas.append(
#         100 * np.mean(damaged > clean)
#     )

# print("Validation foreground erased:")
# print("Average:", round(np.mean(val_erased_fractions), 2), "%")
# print("Median:", round(np.median(val_erased_fractions), 2), "%")

# print("\nValidation damaged area:")
# print("Average:", round(np.mean(val_damaged_areas), 2), "%")

In [ ]:
# category_counts = {
#     "Erased foreground": 0,
#     "Erased background": 0,
#     "Unchanged foreground": 0,
#     "Unchanged background": 0
# }

# total_pixels = 0

# for _ in range(100):
#     image_path = random.choice(train_files)
#     damaged, clean = make_training_pair(image_path)

#     foreground = clean[..., 0] < (128 / 255)
#     erased = damaged[..., 0] > clean[..., 0]

#     category_counts["Erased foreground"] += np.sum(foreground & erased)
#     category_counts["Erased background"] += np.sum(~foreground & erased)
#     category_counts["Unchanged foreground"] += np.sum(foreground & ~erased)
#     category_counts["Unchanged background"] += np.sum(~foreground & ~erased)

#     total_pixels += clean.shape[0] * clean.shape[1]

# for category, count in category_counts.items():
#     print(f"{category:25s}: {100 * count / total_pixels:.3f}%")

In [ ]:
# erased_fractions = []
# erased_areas = []

# for _ in range(500):
#     image_path = random.choice(train_files)
#     damaged, clean = make_training_pair(image_path)

#     original_lines = clean[..., 0] < (128 / 255)
#     erased_lines = original_lines & (damaged[..., 0] >= 0.999)

#     total_lines = np.sum(original_lines)
#     erased = np.sum(erased_lines)

#     if total_lines > 0:
#         erased_fractions.append(100 * erased / total_lines)

#     erased_areas.append(
#         100 * np.mean(damaged[..., 0] > clean[..., 0])
#     )

# print("Average fraction of original line pixels erased:",
#       round(np.mean(erased_fractions), 2), "%")

# print("Median fraction erased:",
#       round(np.median(erased_fractions), 2), "%")

# print("Average damaged area:",
#       round(np.mean(erased_areas), 2), "%")

In [ ]:
# import matplotlib.pyplot as plt

# image_path = random.choice(train_files)
# damaged, clean = make_training_pair(image_path)

# plt.figure(figsize=(10, 5))

# plt.subplot(1, 2, 1)
# plt.imshow(damaged[..., 0], cmap="gray", vmin=0, vmax=1)
# plt.title("Damaged input")
# plt.axis("off")

# plt.subplot(1, 2, 2)
# plt.imshow(clean[..., 0], cmap="gray", vmin=0, vmax=1)
# plt.title("Clean target")
# plt.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
# category_counts = {
#     "Erased foreground": 0,
#     "Erased background": 0,
#     "Unchanged foreground": 0,
#     "Unchanged background": 0
# }

# total_pixels = 0
# erased_fractions = []

# for _ in range(500):
#     image_path = random.choice(train_files)
#     damaged, clean = make_training_pair(image_path)

#     foreground = clean[..., 0] < (128 / 255)
#     erased = damaged[..., 0] > clean[..., 0]

#     category_counts["Erased foreground"] += np.sum(foreground & erased)
#     category_counts["Erased background"] += np.sum(~foreground & erased)
#     category_counts["Unchanged foreground"] += np.sum(foreground & ~erased)
#     category_counts["Unchanged background"] += np.sum(~foreground & ~erased)

#     total_pixels += clean.shape[0] * clean.shape[1]

#     total_lines = np.sum(foreground)
#     if total_lines:
#         erased_fractions.append(
#             100 * np.sum(foreground & erased) / total_lines
#         )

# print("PIXEL DISTRIBUTION")
# for category, count in category_counts.items():
#     print(f"{category:25s}: {100 * count / total_pixels:.3f}%")

# print("\nFOREGROUND ERASURE")
# print("Mean:", round(np.mean(erased_fractions), 2), "%")
# print("Median:", round(np.median(erased_fractions), 2), "%")

In [ ]:
# print("Model1 weighted checkpoint:",
#       find_latest_checkpoint("Model1_weighted"))

# print("Model2 weighted checkpoint:",
#       find_latest_checkpoint("Model2_weighted"))